# 06 - Chat Gradio con memoria SQLite

Este notebook muestra como funciona la app local de chat con memoria persistente. La idea es entender el flujo sin esconderlo demasiado:

- primero usamos `ChatMemoryStore` directamente para ver que se guarda en SQLite;
- luego renderizamos el historial que se manda al LLM;
- despues definimos una funcion explicita de respuesta, parecida a la que usa Gradio;
- al final usamos la abstraccion real `build_app()` de `apps/gradio_chat.py`.

Por defecto este notebook no hace llamadas live al LLM y no abre un servidor Gradio. Puedes activar esas partes cambiando las banderas indicadas.

## Configuracion

Los parametros importantes son:

- `MODEL_ID`: modelo que se pasa a `LLMFactory.create(...)`.
- `MEMORY_TURNS`: cantidad de turnos recientes que se inyectan en el prompt.
- `SYSTEM_PROMPT`: instrucciones globales del asistente.
- `RUN_LIVE_LLM_CALLS`: valor por defecto para llamadas live desde helpers.
- `RUN_EXPLICIT_DEMO_LIVE`: controla si el ejemplo de `respond_explicit(...)` consume API.
- `LAUNCH_GRADIO_APP`: si esta en `False`, no bloquea el notebook levantando la UI.

In [15]:
from pathlib import Path
import socket
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from llmkit.config.settings import get_settings
from llmkit.llms import LLMFactory
from llmkit.memory import ChatMemoryStore
from llmkit.memory.context import render_memory_prompt

get_settings.cache_clear()
settings = get_settings()

MODEL_ID = settings.default_model
MEMORY_TURNS = 8
RUN_LIVE_LLM_CALLS = False
RUN_EXPLICIT_DEMO_LIVE = False
LAUNCH_GRADIO_APP = False
RESET_DEMO_MEMORY = True

DEMO_DB_PATH = ROOT / "data" / "notebook_06_demo_memory.sqlite3"
SYSTEM_PROMPT = "You are a concise, practical assistant. Use recent conversation only when it helps."

print("Model:", MODEL_ID)
print("Demo DB:", DEMO_DB_PATH)
print("Memory turns:", MEMORY_TURNS)

Model: gpt-5-nano
Demo DB: c:\Users\alejo\Documents\GitHub\LLM\data\notebook_06_demo_memory.sqlite3
Memory turns: 8


## 1. Memoria SQLite sin Gradio

La memoria no depende de Gradio ni del LLM. Es una capa local que guarda conversaciones y mensajes.

El esquema minimo es:

- `conversations`: `id`, `title`, `created_at`, `updated_at`.
- `messages`: `id`, `conversation_id`, `role`, `content`, `created_at`.

Esto permite persistir todo el historial, pero mandar al modelo solo una ventana reciente para controlar costo y contexto.

In [16]:
if RESET_DEMO_MEMORY and DEMO_DB_PATH.exists():
    DEMO_DB_PATH.unlink()

store = ChatMemoryStore(DEMO_DB_PATH)
conversation = store.create_conversation("Notebook memory demo")

store.add_message(conversation.id, "user", "Recuerda que mi proyecto usa SQLite para memoria.")
store.add_message(conversation.id, "assistant", "Entendido. Usaremos SQLite como memoria persistente local.")
store.add_message(conversation.id, "user", "Tambien quiero limitar el contexto a pocos turnos.")
store.add_message(conversation.id, "assistant", "Entonces enviaremos solo una ventana reciente al modelo.")

all_messages = store.get_messages(conversation.id)
[(message.role, message.content) for message in all_messages]

[('user', 'Recuerda que mi proyecto usa SQLite para memoria.'),
 ('assistant', 'Entendido. Usaremos SQLite como memoria persistente local.'),
 ('user', 'Tambien quiero limitar el contexto a pocos turnos.'),
 ('assistant', 'Entonces enviaremos solo una ventana reciente al modelo.')]

## 2. Prompt que recibe el LLM

El LLM no lee SQLite directamente. Primero recuperamos los ultimos mensajes y luego los convertimos en texto. Ese texto se combina con el mensaje actual del usuario.

Esta decision mantiene la API de clientes LLM simple: seguimos llamando `llm.invoke(system=..., user=...)`.

In [17]:
recent_history = store.get_recent_messages(conversation.id, turns=MEMORY_TURNS)
current_message = "Como se construye el prompt con memoria?"

user_prompt = render_memory_prompt(current_message, recent_history)
print(user_prompt)

Recent conversation:
User: Recuerda que mi proyecto usa SQLite para memoria.
Assistant: Entendido. Usaremos SQLite como memoria persistente local.
User: Tambien quiero limitar el contexto a pocos turnos.
Assistant: Entonces enviaremos solo una ventana reciente al modelo.

Current user message: Como se construye el prompt con memoria?


## 3. Funcion explicita de respuesta

Esta funcion muestra el flujo completo sin Gradio:

- limpia el mensaje entrante;
- recupera los ultimos `memory_turns` turnos;
- renderiza el prompt con memoria;
- guarda el mensaje del usuario;
- llama al LLM si `run_live=True`;
- guarda la respuesta del asistente;
- devuelve informacion util para inspeccionar.

Los puntos que normalmente modificarias son `model_id`, `memory_turns`, `system_prompt` y la forma de `render_memory_prompt(...)`.

In [18]:
def respond_explicit(
    message: str,
    *,
    store: ChatMemoryStore,
    conversation_id: int,
    model_id: str = MODEL_ID,
    memory_turns: int = MEMORY_TURNS,
    system_prompt: str = SYSTEM_PROMPT,
    run_live: bool = RUN_LIVE_LLM_CALLS,
) -> dict:
    clean_message = message.strip()
    if not clean_message:
        raise ValueError("message must not be empty")

    recent_history = store.get_recent_messages(conversation_id, turns=memory_turns)
    llm_user_prompt = render_memory_prompt(clean_message, recent_history)

    store.add_message(conversation_id, "user", clean_message)

    if run_live:
        llm = LLMFactory.create(model_id.strip() or None)
        response = llm.invoke(system=system_prompt, user=llm_user_prompt)
        assistant_message = response.content.strip() or "(empty response)"
    else:
        assistant_message = (
            "(demo) No llame al LLM. "
            f"El prompt habria incluido {len(recent_history)} mensajes recientes."
        )

    store.add_message(conversation_id, "assistant", assistant_message)

    return {
        "conversation_id": conversation_id,
        "model_id": model_id,
        "memory_turns": memory_turns,
        "system_prompt": system_prompt,
        "llm_user_prompt": llm_user_prompt,
        "assistant_message": assistant_message,
    }

In [19]:
explicit_conversation = store.create_conversation("Explicit function demo")

first = respond_explicit(
    "Hola, estoy probando memoria persistente.",
    store=store,
    conversation_id=explicit_conversation.id,
    run_live=RUN_EXPLICIT_DEMO_LIVE,
)

second = respond_explicit(
    "Que recuerdas de mi mensaje anterior?",
    store=store,
    conversation_id=explicit_conversation.id,
    run_live=RUN_EXPLICIT_DEMO_LIVE,
)

print("Assistant:", second["assistant_message"])
print("\nPrompt sent to the LLM in the second turn:\n")
print(second["llm_user_prompt"])

Assistant: Recuerdo que querías empezar con memoria persistente y te propuse tres escenarios rápidos:

- En el navegador (localStorage/IndexedDB)
  - localStorage: clave-valor sencillo (setItem/getItem) y durabilidad entre recargas.
  - IndexedDB: para objetos y consultas, asíncrono, más completo.

- Memoria persistente real (PMDK / Intel Optane, Linux)
  - Entorno Linux con DAX y libpmem/libpmemobj.
  - Pasos de alto nivel: montar PMEM, crear/open pool, usar transacciones TX_BEGIN/TX_END, comprobar durabilidad tras fallo.

- Otra opción (archivo único con WAL, SQLite, etc.)
  - SQLite en modo WAL para una solución rápida/prototipo.

También pregunté qué necesitas exactamente para ayudarte mejor (qué escenario usas, objetivo: fallo, rendimiento, durabilidad tras cierre/caída; qué sistema/versión).

¿Quières que te pase ya un snippet listo para copiar? Dime:
- ¿Qué escenario prefieres ahora (navegador, PMDK, u otro)?
- ¿Qué objetivo exacto tienes?
- ¿Qué sistema/versión estás usando?

C

## 4. Version Gradio explicita

La app real usa `apps/gradio_chat.py`, pero esta version reducida deja visibles los callbacks principales.

Partes importantes:

- `model_id`: se puede cambiar desde la UI.
- `conversation_label`: identifica que conversacion de SQLite se esta usando.
- `chat_history`: estado visual de Gradio; no es la fuente de verdad persistente.
- `store`: la fuente de verdad persistente.
- `memory_turns`: controla cuantos mensajes recientes entran al prompt.

In [20]:
def build_chat_app_explicit(
    *,
    store: ChatMemoryStore,
    default_model: str = MODEL_ID,
    memory_turns: int = MEMORY_TURNS,
    system_prompt: str = SYSTEM_PROMPT,
):
    import gradio as gr

    def label_for(conversation):
        return f"{conversation.id}: {conversation.title}"

    def choices():
        conversations = store.list_conversations()
        if not conversations:
            conversations = [store.create_conversation()]
        return [label_for(item) for item in conversations]

    def conversation_id(label):
        if not label:
            return store.create_conversation().id
        return int(label.split(":", 1)[0])

    def to_chatbot_messages(conversation_id_value):
        return [
            {"role": item.role, "content": item.content}
            for item in store.get_messages(conversation_id_value)
        ]

    def load_conversation(label):
        current_id = conversation_id(label)
        return to_chatbot_messages(current_id)

    def answer(message, chat_history, conversation_label, model_id):
        current_id = conversation_id(conversation_label)
        result = respond_explicit(
            message,
            store=store,
            conversation_id=current_id,
            model_id=model_id,
            memory_turns=memory_turns,
            system_prompt=system_prompt,
            run_live=True,
        )
        updated_history = chat_history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": result["assistant_message"]},
        ]
        return "", updated_history

    initial_choices = choices()
    initial_label = initial_choices[0]
    initial_id = conversation_id(initial_label)

    css = """
    .gradio-container { max-width: 1120px !important; margin: 0 auto !important; }
    #chatbot { min-height: 520px; }
    """

    with gr.Blocks(title="LLMKit Chat Notebook") as demo:
        with gr.Row():
            model = gr.Textbox(label="Model", value=default_model, scale=2)
            conversation = gr.Dropdown(
                label="Conversation",
                choices=initial_choices,
                value=initial_label,
                scale=3,
            )

        chatbot = gr.Chatbot(value=to_chatbot_messages(initial_id), elem_id="chatbot")
        message = gr.Textbox(label="Message", lines=3)
        send_button = gr.Button("Send", variant="primary")

        message.submit(
            answer,
            inputs=[message, chatbot, conversation, model],
            outputs=[message, chatbot],
        )
        send_button.click(
            answer,
            inputs=[message, chatbot, conversation, model],
            outputs=[message, chatbot],
        )
        conversation.change(load_conversation, inputs=[conversation], outputs=[chatbot])

    return demo, css


explicit_demo, explicit_css = build_chat_app_explicit(store=store)
type(explicit_demo).__name__

'Blocks'

La funcion anterior es didactica. La app productiva local esta en `apps/gradio_chat.py` y agrega detalles de uso diario: crear conversaciones, limpiar historial, cargar opciones y reutilizar la base `data/chat_memory.sqlite3`.

## 5. Uso de la abstraccion real

Esta es la forma recomendada para usar la app ya creada. `build_app()` encapsula los callbacks de Gradio, la memoria SQLite y el selector de modelo.

No ejecutamos `launch()` por defecto para que el notebook no quede bloqueado durante pruebas automatizadas.

In [24]:
import importlib
import apps.gradio_chat as gradio_chat

# Jupyter cachea modulos importados. reload() fuerza a leer cambios recientes en apps/gradio_chat.py.
gradio_chat = importlib.reload(gradio_chat)

APP_CSS = gradio_chat.APP_CSS
DB_PATH = gradio_chat.DB_PATH
APP_MEMORY_TURNS = gradio_chat.MEMORY_TURNS
SERVER_PORT = gradio_chat.SERVER_PORT
build_app = gradio_chat.build_app

print("App DB:", DB_PATH)
print("App memory turns:", APP_MEMORY_TURNS)
print("App port:", SERVER_PORT)

app = build_app()
type(app).__name__

App DB: C:\Users\alejo\Documents\GitHub\LLM\data\chat_memory.sqlite3
App memory turns: 8
App port: 7860


'Blocks'

In [22]:
def find_available_port(preferred_port: int) -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        if probe.connect_ex(("127.0.0.1", preferred_port)) != 0:
            return preferred_port

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        probe.bind(("127.0.0.1", 0))
        return int(probe.getsockname()[1])


if LAUNCH_GRADIO_APP:
    port = find_available_port(SERVER_PORT)
    if port != SERVER_PORT:
        print(f"Port {SERVER_PORT} is busy. Using available port {port}.")
    app.launch(
        css=APP_CSS,
        server_name="127.0.0.1",
        server_port=port,
        prevent_thread_lock=True,
        inbrowser=True,
    )
else:
    print("Set LAUNCH_GRADIO_APP = True para abrir la UI desde el notebook.")
    print("Alternativa desde terminal: python apps/gradio_chat.py")

OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.

## Que puedes modificar

- Cambia `MEMORY_TURNS` si quieres mandar mas o menos historial al modelo.
- Cambia `SYSTEM_PROMPT` para controlar personalidad, idioma, reglas o formato de respuesta.
- Cambia `DEMO_DB_PATH` o la variable de entorno `LLMKIT_CHAT_MEMORY_DB` para usar otra base SQLite.
- Cambia `MODEL_ID` o el textbox `Model` en Gradio para probar otro proveedor soportado por `LLMFactory`.
- Cambia `render_memory_prompt(...)` si quieres otro formato de historial, por ejemplo XML, JSON o secciones con timestamps.
- Agrega resumen automatico si el historial crece demasiado; en esta version solo se usa ventana reciente.